<a href="https://colab.research.google.com/github/mducdaf2/ai-knowledges/blob/main/NLP/Classification_VectorSpaces/Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis with Naive Bayes

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
from pathlib import Path
import re
import string

import nltk
from nltk.corpus import stopwords, twitter_samples
from nltk.tokenize import TweetTokenizer
from nltk.stem import PorterStemmer

## 1. Process Data

In [3]:
nltk.download('twitter_samples')
nltk.download('stopwords')

[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Unzipping corpora/twitter_samples.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
# get the sets of positive and negative tweets
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

# split the data into two pieces, one for training and one for testing (validation set)
test_pos = all_positive_tweets[4000:]
train_pos = all_positive_tweets[:4000]
test_neg = all_negative_tweets[4000:]
train_neg = all_negative_tweets[:4000]

train_x = train_pos + train_neg
test_x = test_pos + test_neg

# avoid assumptions about the length of all_positive_tweets
train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))
test_y = np.append(np.ones(len(test_pos)), np.zeros(len(test_neg)))

print(len(train_x), len(train_y))

8000 8000


In [37]:
print(train_x[0])
print(train_y[0])

#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)
1.0


### 1.1. Convert tweet to clean word.

In [5]:
def process_text(tweet):
    """Process tweet function.
    Input:
        tweet: a string containing a tweet
    Output:
        tweets_clean: a list of words containing the processed tweet

    """
    stemmer = PorterStemmer()
    stopwords_en = stopwords.words('english')

    # remove stock market tickers like $GE
    tweet = re.sub(r'\$\w*', '', tweet)
    # remove old style retweet text "RT"
    tweet = re.sub(r'^RT[\s]+', '', tweet)
    # remove hyperlinks
    tweet = re.sub(r'https?://[^\s\n\r]+', '', tweet)
    # remove hashtags
    # only removing the hash # sign from the word
    tweet = re.sub(r'#', '', tweet)

    # Tokenize
    tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
    tweet_tokens = tokenizer.tokenize(tweet)

    tweets_clean = []
    for word in tweet_tokens:
        if (word not in stopwords_en and  # remove stopwords
                word not in string.punctuation):  # remove punctuation
            # tweets_clean.append(word)
            stem_word = stemmer.stem(word)  # stemming word
            tweets_clean.append(stem_word)
    return tweets_clean

In [9]:
custom_tweet = "RT @Twitter @chapagain Hello There! Have a great day. :) #good #morning http://chapagain.com.np"

# print cleaned tweet
print(process_text(custom_tweet))

['hello', 'great', 'day', ':)', 'good', 'morn']


### 1.2. Count tweets
Tạo dictionary chứa các cặp `(clean_word, sentiment): freqs`

In [10]:
def count_tweets(result, tweets, sentiments):
    for y, tweet in zip(sentiments, tweets):
        for word in process_text(tweet):
            pair = (word, y)

            result[pair] = result.get(pair, 0) + 1

    return result

In [11]:
result = {}
tweets = ['i am happy', 'i am tricked', 'i am sad', 'i am tired', 'i am tired']
ys = [1, 0, 0, 0, 0]
count_tweets(result, tweets, ys)

{('happi', 1): 1, ('trick', 0): 1, ('sad', 0): 1, ('tire', 0): 2}

## Naive Bayes

Muốn train một Naive Bayes classifier, đầu tiên chúng ta cần xác định số lượng class. Bằng việc tính xác suất cho mỗi class, trong bài này ta sẽ tính:
$$ P(D_\text{pos}) = \frac{D_\text{pos}}{D} $$

$$ P(D_\text{neg}) = \frac{D_\text{neg}}{D} $$

Trong đó:
- $D$ là số lượng tweets
- $D_\text{pos}$ là số lượng positive tweets
- $D_\text{neg}$ là số lượng negative tweets

**Prior** được hiểu là xác suất của positive so với negative, được tính bằng:
$$ \text{prior} = \frac{P(D_\text{pos})}{P(D_\text{neg})} $$

Thực hiện lấy log để rescale nó xuống, ta thu được:
$$ \text{logprior} = \log\left( \frac{P(D_\text{pos})}{P(D_\text{neg})} \right)
= \log\left( \frac{D_\text{pos}}{D_\text{neg}} \right)
= \log(D_\text{pos}) - \log(D_\text{neg}) $$

Tiếp theo, tính xác suất mỗi từ là positive hay negative:
$$ P(W_\text{pos}) = \frac{freq_{pos} + 1}{N_{pos} + V} $$
$$ P(W_\text{neg}) = \frac{freq_{neg} + 1}{N_{neg} + V} $$

Trong đó:
- $N_{pos}, N_{neg}$ là tổng số postive và negative words trong tất cả documents.
- $V$ là số từ độc lập (unique words) trong tập documents, không phân biệt sentiment, đây được hiểu như vocabulary.
- Cộng thêm `+1` ở tử để tránh việc xác suất bằng 0, kỹ thuật được gọi là Laplace smoothing.

Cuối cùng, chúng ta tính `log-likelihood`, được hiểu là khả năng của từ đó:
$$ \text{loglikelihood} = \log\left( \frac{P(W_{pos})}{P(W_{neg})} \right) $$

### 2.1. Build frequencies dictionary

In [31]:
freqs = count_tweets({}, train_x, train_y)

In [17]:
print(freqs[('happi', 1)])
print(freqs[('happi', 0)])

162
18


### 2.2. Train Naive-Bayes

In [40]:
def train_naive_bayes(freqs, train_x, train_y):
    '''
    Input:
        freqs: dictionary from (word, label) to how often the word appears
        train_x: a list of tweets
        train_y: a list of labels corresponding to the tweets (0,1)
    Output:
        logprior: the log prior.
        loglikelihood: the log likelihood of you Naive bayes equation. (equation 6 above)
    '''
    loglikelihood = {}
    logprior = 0

    # calculate V, the number of unique words in the vocabulary
    vocab = set([pair[0] for pair in freqs.keys()])
    V = len(vocab)

    # calculate N_pos, N_neg, V_pos, V_neg
    N_pos = N_neg = 0
    for pair in freqs.keys():
        if pair[1] > 0:     # if the label is positive (greater than zero)
            N_pos += freqs[pair]
        else:               # else, the label is negative
            N_neg += freqs[pair]

    # Calculate D, the number of documents
    D = len(train_y)

    # Calculate D_pos, the number of positive documents
    D_pos = (len(list(filter(lambda x: x > 0, train_y))))

    # Calculate D_neg, the number of negative documents
    D_neg = (len(list(filter(lambda x: x <= 0, train_y))))

    # Calculate logprior
    logprior = np.log(D_pos) - np.log(D_neg)

    # For each word in the vocabulary...
    for word in vocab:
        # get the positive and negative frequency of the word
        freq_pos = freqs.get((word, 1), 0)
        freq_neg = freqs.get((word, 0), 0)

        # calculate the probability that each word is positive, and negative
        p_w_pos = (freq_pos + 1) / (N_pos + V)
        p_w_neg = (freq_neg + 1) / (N_neg + V)

        # calculate the log likelihood of the word
        loglikelihood[word] = np.log(p_w_pos/p_w_neg)

    return logprior, loglikelihood

In [41]:
logprior, loglikelihood = train_naive_bayes(freqs, train_x, train_y)
print(logprior)
print(len(loglikelihood))

0.0
9143


In [45]:
print(loglikelihood['happi'])
print(loglikelihood["sad"])

2.1349587642535863
-2.83771350499994


### 2.3. Predict using Naive-Bayes

Với mỗi tweet chúng ta cần tính được xác suất tweet đó thuộc positive hay negative sentiment. Ta thu được giá trị predicted sentiment như sau:
$$ p = logprior + \sum_i^N (loglikelihood_i) $$

Công thức này xuất phát từ việc so sánh 2 xác suất: $P(pos | tweet)$ và $P(neg | tweet)$. Sử dụng định lý Bayes, ta có:
$$ \frac{P(pos | tweet)}{P(neg | tweet)} = \frac{P(tweet | pos)}{P(tweet | neg)} \times \frac{P(pos)}{P(neg)} $$

Với giả định ngây thơ (Naive): giả sử các từ trong tweet độc lập với nhau, khi đó ta có:
$$ P(tweet | pos)=P(w_1​, w_2​,...,w_N ​∣ pos) \simeq \prod_i^N ​P(w_i ​∣ pos) $$

Kết hợp lại, lấy log ta sẽ thu được công thức phía đầu.

In [46]:
def naive_bayes_predict(tweet, logprior, loglikelihood):
    '''
    Input:
        tweet: a string
        logprior: a number
        loglikelihood: a dictionary of words mapping to numbers
    Output:
        p: the sum of all the logliklihoods of each word in the tweet (if found in the dictionary) + logprior (a number)

    '''
    word_l = process_text(tweet)

    # initialize probability to zero
    p = 0

    # add the logprior
    p += logprior

    for word in word_l:
        if word in loglikelihood:
            p += loglikelihood[word]
    return p

In [47]:
my_tweet = 'She smiled.'
p = naive_bayes_predict(my_tweet, logprior, loglikelihood)
print('The expected output is', p)

The expected output is 1.5542634605271097


### 2.4. Test model
Kiểm tra accuracy trên tập test

In [48]:
def test_naive_bayes(test_x, test_y, logprior, loglikelihood, naive_bayes_predict=naive_bayes_predict):
    """
    Input:
        test_x: A list of tweets
        test_y: the corresponding labels for the list of tweets
        logprior: the logprior
        loglikelihood: a dictionary with the loglikelihoods for each word
    Output:
        accuracy: (# of tweets classified correctly)/(total # of tweets)
    """
    accuracy = 0
    y_hats = []
    for tweet in test_x:
        if naive_bayes_predict(tweet, logprior, loglikelihood) > 0:
            y_hat_i = 1
        else:
            y_hat_i = 0
        y_hats.append(y_hat_i)

    # error is the average of the absolute values of the differences between y_hats and test_y
    error = np.mean(np.absolute(y_hats-test_y))

    # Accuracy is 1 minus the error
    accuracy = 1-error
    return accuracy

In [49]:
print("Naive Bayes accuracy = %0.4f" %
      (test_naive_bayes(test_x, test_y, logprior, loglikelihood)))

Naive Bayes accuracy = 0.9955


In [52]:
for tweet in ['I am happy', 'I am bad', 'this movie should have been great.', 'great', 'great great', 'great great great', 'great great great great']:
    p = naive_bayes_predict(tweet, logprior, loglikelihood)
    print(f'{tweet} -> {p:.2f}')

I am happy -> 2.13
I am bad -> -1.31
this movie should have been great. -> 2.11
great -> 2.13
great great -> 4.25
great great great -> 6.38
great great great great -> 8.50
